# ANON AI ULTRA — Türkçe devam eğitimi + sohbet ince ayarı

Bu defter, **senin sıfırdan eğittiğin ANON temel modelini** geliştirir. Harici bir hazır modelin ağırlıklarını indirmez. İlk olarak daha fazla Türkçe metinle devam eğitimi yapar; ardından Apache-2.0 lisanslı olarak belirtilen Türkçe sohbet veri kümesinin `tr_only` bölümünü filtreleyip ince ayar uygular.

**Ön koşul:** Önceki eğitimden indirdiğin `ANON-AI-ULTRA-base-30m.zip` dosyası. Ücretsiz GPU süresi kesilebilir; uzun eğitim ücretsiz Colab'da garanti edilmez. Eğitim tamamlanınca ZIP çıktılarını indir.

In [ ]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU bulunamadı')
assert torch.cuda.is_available(), 'Colab > Runtime > Change runtime type bölümünden GPU seç.'
WORK_DIR = '/content'
print('Çalışma klasörü:', WORK_DIR)

In [ ]:
import os, subprocess, sys
REPO_DIR = os.path.join(WORK_DIR, 'ANON-AI-ULTRA')
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/furkanmeyra2173-create/ANON-AI-ULTRA.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'training/requirements.txt'], check=True)
print('ANON AI eğitim kodları hazır.')

## 1. Önceki ANON temel modelini içe aktar

ZIP dosyası bu Colab oturumunda yoksa telefondan yükle. Dosya yapısında `config.json` bulunmalı.

In [ ]:
from pathlib import Path
import zipfile, shutil
BASE_ZIP = os.path.join(WORK_DIR, 'ANON-AI-ULTRA-base-30m.zip')
if not os.path.isfile(BASE_ZIP):
    from google.colab import files
    print('Önceki eğitimden indirdiğin ANON-AI-ULTRA-base-30m.zip dosyasını seç.')
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith('.zip')]
    assert candidates, 'Bir ZIP dosyası yüklenmedi.'
    BASE_ZIP = os.path.abspath(candidates[0])
BASE_IMPORT_DIR = os.path.join(WORK_DIR, 'anon-ai-base-import')
shutil.rmtree(BASE_IMPORT_DIR, ignore_errors=True)
os.makedirs(BASE_IMPORT_DIR, exist_ok=True)
with zipfile.ZipFile(BASE_ZIP) as zf:
    zf.extractall(BASE_IMPORT_DIR)
configs = list(Path(BASE_IMPORT_DIR).rglob('config.json'))
assert configs, 'ZIP içinde config.json bulunamadı; doğru temel model ZIP dosyasını yükle.'
BASE_MODEL_DIR = str(configs[0].parent)
print('Temel model bulundu:', BASE_MODEL_DIR)

## 2. Türkçe metinle devam eğitimi

Mevcut tokenizer ve ANON ağırlıkları korunur. Model yeniden rastgele başlatılmaz. İlk deneme 50.000 belge sınırı ve 1.200 ek eğitim adımı kullanır.

In [ ]:
CONTINUED_DIR = os.path.join(WORK_DIR, 'anon-ai-ultra-base-continued')
command = [sys.executable, 'training/continue_pretraining.py',
    '--base-model', BASE_MODEL_DIR,
    '--dataset-id', 'Ethosoft/Turkish_corpus', '--split', 'train',
    '--text-column', 'text', '--max-documents', '50000',
    '--max-steps', '1200', '--block-size', '512',
    '--output-dir', CONTINUED_DIR]
subprocess.run(command, check=True)
CONTINUED_MODEL_DIR = os.path.join(CONTINUED_DIR, 'final')
assert os.path.isdir(CONTINUED_MODEL_DIR), 'Devam eğitimi modeli kaydedilemedi.'
print('Devam eğitimi tamamlandı:', CONTINUED_MODEL_DIR)

## 3. Türkçe sohbet verisini hazırla

Varsayılan filtre yalnızca `language_mode=tr_only`, `prompt_language=tr` ve araç çağrısı içermeyen örnekleri alır. Kaynak veri kümesinin kartında Apache-2.0 lisansı belirtilir; çıktı yanında kaynak/lisans metaverisi de saklanır.

In [ ]:
SFT_JSONL = os.path.join(WORK_DIR, 'turkish_chat_train.jsonl')
command = [sys.executable, 'training/prepare_instruction_data.py',
    '--dataset-id', 'cisimcik/turkish-chat-max-25k', '--config', 'all',
    '--split', 'train', '--language-mode', 'tr_only', '--prompt-language', 'tr',
    '--max-samples', '20000', '--output', SFT_JSONL]
subprocess.run(command, check=True)
print('JSONL hazır:', SFT_JSONL)
print('Satır sayısı:', sum(1 for _ in open(SFT_JSONL, encoding='utf-8')))

## 4. Sohbet ince ayarı (SFT)

Kayıp yalnızca asistan yanıtlarında hesaplanır. Verinin %5'i doğrulama için ayrılır; en iyi doğrulama kaydı son model olarak kaydedilir.

In [ ]:
CHAT_DIR = os.path.join(WORK_DIR, 'anon-ai-ultra-chat')
command = [sys.executable, 'training/finetune_instructions.py',
    '--base-model', CONTINUED_MODEL_DIR,
    '--dataset-jsonl', SFT_JSONL,
    '--output-dir', CHAT_DIR,
    '--max-samples', '20000', '--max-length', '512',
    '--max-steps', '800', '--eval-ratio', '0.05',
    '--eval-steps', '50', '--batch-size', '2',
    '--gradient-accumulation-steps', '8', '--learning-rate', '2e-5']
subprocess.run(command, check=True)
CHAT_MODEL_DIR = os.path.join(CHAT_DIR, 'final')
assert os.path.isdir(CHAT_MODEL_DIR), 'Sohbet modeli kaydedilemedi.'
print('Sohbet modeli hazır:', CHAT_MODEL_DIR)

## 5. İlk sohbet testi

Bu test, modelin mesaj biçimini okuyup yanıt ürettiğini kontrol eder. Yanıtın doğru veya kaliteli olduğu varsayılmaz; çıktıyı ayrıca değerlendireceğiz.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert os.path.isdir(CHAT_MODEL_DIR), f'Sohbet modeli bulunamadı: {CHAT_MODEL_DIR}'
test_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL_DIR)
test_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL_DIR)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
test_model.to(device)
test_model.eval()
test_model.eval()

print("Tokenizer vocab:", len(test_tokenizer))
print("Model embedding rows:", test_model.get_input_embeddings().num_embeddings)
assert len(test_tokenizer) == test_model.get_input_embeddings().num_embeddings, "Tokenizer/model vocab mismatch."
assert test_tokenizer.eos_token_id is not None and test_tokenizer.pad_token_id is not None, "EOS/PAD token is missing."
for marker in ["<|system|>", "<|user|>", "<|assistant|>"]:
    marker_ids = test_tokenizer.encode(marker, add_special_tokens=False)
    assert len(marker_ids) == 1 and marker_ids[0] != test_tokenizer.unk_token_id, f"Invalid chat marker: {marker}"
probe = "Türkçe karakter testi: ğüşiöç İı"
assert test_tokenizer.decode(test_tokenizer.encode(probe, add_special_tokens=False), skip_special_tokens=False) == probe, "Tokenizer Turkish round-trip failed."
print("Tokenizer, chat markers and model dimensions: OK")

test_questions = [
    'Merhaba, nasılsın?',
    'Türkiye\'nin başkenti neresidir?',
    'Yapay zekâ nedir?',
    'Almanca merhaba nasıl söylenir?',
    'Bana kendini kısaca tanıtır mısın?'
]

print('ANON AI sohbet testi başlıyor')
print('Çalışan cihaz:', device)
print('=' * 60)

for index, question in enumerate(test_questions, start=1):
    prompt = '<|user|>' + question + test_tokenizer.eos_token + '<|assistant|>'
    inputs = test_tokenizer(prompt, return_tensors='pt').to(device)
    with torch.inference_mode():
        output = test_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=test_tokenizer.pad_token_id,
            eos_token_id=test_tokenizer.eos_token_id
        )
    new_tokens = output[0, inputs['input_ids'].shape[1]:]
    answer = test_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(f'\nTEST {index}: {question}')
    print('ANON AI:', answer if answer else '[Boş yanıt]')
    print('-' * 60)

print('Test tamamlandı. Cevapların kalitesi ayrıca değerlendirilmeli; hücrenin çalışması tek başına başarılı sohbet anlamına gelmez.')


## 6. Model dosyasını ZIP olarak hazırla

ZIP'i indirip sakla. Bu deneysel modeli Android uygulamasına hemen bağlama; önce sohbet çıktıları, doğruluk, tekrar, güvenlik ve cihaz belleği test edilmeli.

In [ ]:
CHAT_ZIP = os.path.join(WORK_DIR, 'ANON-AI-ULTRA-chat.zip')
with zipfile.ZipFile(CHAT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file in Path(CHAT_MODEL_DIR).rglob('*'):
        if file.is_file():
            archive.write(file, file.relative_to(CHAT_MODEL_DIR))
print('İndirmeye hazır:', CHAT_ZIP)
from google.colab import files
files.download(CHAT_ZIP)